# V8 ALL 一次启动四个验证状态

选择 T4 GPU → 保存 → 全部运行。无需再手动每两个分支点一次。

默认开启 SAVE_TO_DRIVE：首次需要授权挂载自己的 Google Drive。每完成一个分支备份检查点，四状态结果保存在 MyDrive/CAPE_TTA_V8_Validation。

若新运行时没有本地结果或 Drive 备份，会提示上传之前的 task-1-init-0.zip，已有的两个基线分支不会重复运行。保留 RESTORE_PREVIOUS_ZIP 默认勾选。

任务为 (task 1, init 0)、(1,1)、(2,0)、(2,1)。每个不同候选完整运行，包含随机选择的有限池期望和短期评分选择对照。每状态最多九个不同动作加一次基线回放，默认一次跑完该状态全部剩余分支。

四状态历史粗估上限约 6–9 小时，实际视重复率、早停与环境而变。免费 GPU 不保证连续资源；中断后可再次运行，从 Drive 备份恢复，未完成的单个分支需要重放。不要使用自动点击或绕过资源限制。

使用既有实验代码版本 998ffd17f9ad344088b8e98b53bbe8da5ff9626c，兼容之前的 V8 包。只新增 host 侧调度和备份，不修改候选规则。依赖或模型代码哈希变化时会拒绝混合续跑结果。

只有 all_four_validation_cases_complete 表示四个验证状态跑完。GPU 调度尚未实际验证；源码、汇总与恢复模块已通过本地检查。该流程不包含尚未设计完成的最终主实验或 LoRA，不能据此声称 CCF A 已达标。


In [ ]:
# One launch executes all four V8 validation cases; no two-branch pauses.
SAVE_TO_DRIVE = True #@param {type:"boolean"}
RESTORE_PREVIOUS_ZIP = True #@param {type:"boolean"}
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import traceback
import uuid

EXPERIMENT_SHA = "998ffd17f9ad344088b8e98b53bbe8da5ff9626c"
SUPPORT_SHA = "8d75c567892f63de5829c8a6c831515ef3ede1e8"
SOURCE = Path("/content/cape-source") / EXPERIMENT_SHA
SUPPORT = Path("/content/cape-support") / SUPPORT_SHA
ROOT = Path("/content/cape-ranking-v8")
ROOT.mkdir(parents=True, exist_ok=True)
PY = Path("/content/cape-env-v3/bin/python")
LIBERO = Path("/content/LIBERO-8f1084e3132a")
EXPORT = Path("/content/CAPE_TTA_V8_ALL.zip")
state = {"status":"running", "scope":"four_case_validation_only", "experiment_sha":EXPERIMENT_SHA, "completed_cases":[]}
log = (ROOT / "all_launcher.log").open("a", encoding="utf-8")
helper = None
mirror = None

def write_state():
    (ROOT / "all_status.json").write_text(json.dumps(state, indent=2), encoding="utf-8")

def say(line):
    print(line, flush=True); log.write(line + "\n"); log.flush()

def run(command, env=None):
    say("$ " + " ".join(map(str, command)))
    with subprocess.Popen(list(map(str, command)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as proc:
        for line in proc.stdout:
            print(line, end="", flush=True); log.write(line); log.flush()
        rc=proc.wait()
    if rc: raise subprocess.CalledProcessError(rc,command)

def checkout(path,sha):
    if not path.exists():
        path.mkdir(parents=True)
        run(["git","-C",path,"init"])
        run(["git","-C",path,"remote","add","origin","https://github.com/weichezhang/CAPE-TTA.git"])
        run(["git","-C",path,"fetch","--depth","1","origin",sha])
        run(["git","-C",path,"checkout","--detach",sha])
    actual=subprocess.check_output(["git","-C",str(path),"rev-parse","HEAD"],text=True).strip()
    if actual!=sha: raise RuntimeError("Source revision mismatch")
    subprocess.run(["git","-C",str(path),"diff","--quiet","HEAD"],check=True)

def persist():
    write_state()
    if mirror is not None and helper is not None:
        for t,s in helper.CASES:
            helper.checkpoint(ROOT/f"task-{t}-init-{s}"/"case",mirror/f"task-{t}-init-{s}.zip")
        helper.export_all(ROOT,mirror/"latest_all.zip")
        say("CHECKPOINT 已保存到 Google Drive")

def run_case(command,case,env):
    # Model is loaded once per case; completed branches trigger persistent backups.
    previous=json.loads((case/"audit.json").read_text()) if (case/"audit.json").is_file() else {}
    backed=len(previous.get("branches",{}))
    say("$ " + " ".join(map(str,command)))
    proc=subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
    try:
        for line in proc.stdout:
            print(line,end="",flush=True);log.write(line);log.flush()
            if (case/"audit.json").is_file():
                current=json.loads((case/"audit.json").read_text())
                finished=len(current.get("branches",{}))
                if finished>backed:
                    persist();backed=finished
        rc=proc.wait()
        if rc:raise subprocess.CalledProcessError(rc,command)
    finally:
        if proc.poll() is None:
            proc.terminate()
            try:proc.wait(timeout=10)
            except subprocess.TimeoutExpired:proc.kill();proc.wait()
        persist()

try:
    checkout(SOURCE,EXPERIMENT_SHA)
    checkout(SUPPORT,SUPPORT_SHA)
    spec=importlib.util.spec_from_file_location("v8_all_support",SUPPORT/"scripts/v8_all_support.py")
    helper=importlib.util.module_from_spec(spec);spec.loader.exec_module(helper)
    if SAVE_TO_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        mirror=Path("/content/drive/MyDrive/CAPE_TTA_V8_Validation")
        mirror.mkdir(parents=True,exist_ok=True)
    has_local=any((ROOT/f"task-{t}-init-{s}"/"case/audit.json").is_file() for t,s in helper.CASES)
    if not has_local and mirror is not None and (mirror/"latest_all.zip").is_file():
        restored=helper.restore(mirror/"latest_all.zip",ROOT,SOURCE)
        say("已从 Drive 自动恢复："+str(restored));has_local=True
    if not has_local and RESTORE_PREVIOUS_ZIP:
        from google.colab import files
        say("请选择之前已完成两个基线分支的 task-1-init-0.zip，或上次 V8 ALL ZIP。")
        uploaded=files.upload()
        if len(uploaded)!=1:raise RuntimeError("请只上传一个 V8 结果 ZIP")
        archive=ROOT/"import.zip";archive.write_bytes(next(iter(uploaded.values())))
        restored=helper.restore(archive,ROOT,SOURCE);archive.unlink()
        say("已恢复，已有分支不会重跑："+str(restored))
    summaries=[]
    for t,s in helper.CASES:
        case=ROOT/f"task-{t}-init-{s}"/"case"
        if (case/"audit.json").is_file():
            completed=helper.complete_case(case,t,s,SOURCE)
            if completed is not None:summaries.append(completed)
    state["completed_cases"]=[{"task_id":x["task_id"],"init_state_id":x["init_state_id"]} for x in summaries]
    persist()
    if len(summaries)<4:
        smi=shutil.which("nvidia-smi")
        if smi is None:raise RuntimeError("需要 T4 GPU；当前没有可用 GPU。")
        run([smi,"--query-gpu=name,memory.total","--format=csv,noheader"])
        warm=False
        if PY.is_file() and LIBERO.is_dir():
            check=subprocess.run([str(PY),"-c","import torch,transformers,bitsandbytes,mujoco; assert torch.cuda.is_available(); assert torch.__version__.startswith('2.2.0'); assert mujoco.__version__=='2.3.7'"],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
            warm=check.returncode==0;log.write(check.stdout);log.flush()
        if not warm:
            run([sys.executable,"-u",SOURCE/"scripts/colab_bootstrap.py","--prepare-only","--run-dir",ROOT/("setup-"+uuid.uuid4().hex[:8])])
        run([PY,"-m","pip","install","--no-build-isolation","--no-deps","-e",SOURCE])
        env=dict(os.environ,HF_HOME="/content/cape-hf-cache",PYTHONUNBUFFERED="1",MUJOCO_GL="osmesa",PYOPENGL_PLATFORM="osmesa",TF_CPP_MIN_LOG_LEVEL="2")
        for t,s in helper.CASES:
            case=ROOT/f"task-{t}-init-{s}"/"case"
            done=helper.complete_case(case,t,s,SOURCE) if (case/"audit.json").is_file() else None
            if done is not None:
                say(f"SKIP task={t} init={s} 已完成")
                continue
            state.update(current_task=t,current_initial_state=s);persist()
            command=[PY,"-u",SOURCE/"scripts/run_ranking_audit_v8.py","--libero-root",LIBERO,"--run-dir",case,"--task-id",str(t),"--init-state-id",str(s),"--max-new-branches","10"]
            if (case/"audit.json").is_file():command.append("--resume")
            run_case(command,case,env)
            done=helper.complete_case(case,t,s,SOURCE)
            if done is None:raise RuntimeError("Case did not complete")
            summaries.append(done)
            state["completed_cases"]=[{"task_id":x["task_id"],"init_state_id":x["init_state_id"]} for x in summaries]
            persist();say(f"COMPLETE task={t} init={s}")
    state["status"]="all_four_validation_cases_complete"
    (ROOT/"validation_summary.json").write_text(json.dumps({"scope":"four_validation_cases_not_final_test", "cases":summaries},indent=2))
    persist();say("四个验证状态全部完成。请上传最终 ZIP；这不是整篇论文全部正式实验完成。")
except BaseException as exc:
    state.update(status="failed_or_interrupted",error={"type":type(exc).__name__,"message":str(exc),"traceback":traceback.format_exc()})
    say(traceback.format_exc())
    raise
finally:
    write_state()
    if helper is not None:
        try:
            persist()
            helper.export_all(ROOT,EXPORT)
            say("最终或中断结果包："+str(EXPORT))
        except Exception:
            say("备份失败："+traceback.format_exc())
    log.close()
    if EXPORT.is_file():
        from google.colab import files
        files.download(str(EXPORT))
